# D5 · Threshold calibration

Reproduces the statistical calibration behind detector D2 and the acceptance report (`python -m qsentinel.acceptance`).
Adapted from Anansh Jain's D1–D6 notebook to the integrated package: the exact binomial tail is the operating probability; Chernoff and Hoeffding are provable ceilings, so they are deliberately **not** treated as equal.

In [ ]:
from qsentinel.detect.calibrate import forgery_bound, forgery_exact, hoeffding_threshold, tradeoff_table
from qsentinel.detect.validate import false_alarm_validation, monte_carlo_forgery

## 1. Forger's chance per block
Six-state encoding: a forger mismatches each round with probability q = 1/3. Default n = 256, τ = 10%.

In [ ]:
print('exact   ', forgery_exact(256, 0.10, 1/3))
print('Chernoff', forgery_bound(256, 0.10, 1/3))
print('(3/4)^128 =', forgery_bound(128, 0.0, 0.25))
tradeoff_table()

## 2. Closed-form Hoeffding threshold
$p_{th} = p + \sqrt{\ln(1/lpha) / (2n)}$ for baseline mismatch $p$ and false-alarm budget $lpha$.

In [ ]:
for n in (64, 128, 256, 512):
    print(n, round(hoeffding_threshold(n, 0.02, 1e-6), 4))

## 3. 100,000-trial false-alarm validation
Honest blocks on a 4% noisy channel, n = 64, τ = 10%: the exact tail must sit inside the 95% Wilson interval of the simulated rate.

In [ ]:
rep = false_alarm_validation(64, 0.10, 0.04, trials=100_000, seed=2026)
rep

## 4. Forgery bound through the quantum simulator
A measure-a-copy forger run through the Stim backend; D5 requires ≤ 10% relative error.

In [ ]:
monte_carlo_forgery(16, 0.10, 40_000, seed=0)